In [41]:
# !pip install datasets==3.6.0
# !pip uninstall -y torchao
# !pip install torchao==0.16.0 --no-cache-dir
# !pip uninstall -y torchvision
# !pip install torchvision==0.23.0 --no-cache-dir

In [42]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

from datasets import Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import TrainingArguments, Trainer

from peft import get_peft_model, LoraConfig, TaskType

In [43]:
reviews = pd.read_csv("imdb.csv").sample(500)
reviews.head()

,review,sentiment
6252,"When I first popped in Happy Birthday to Me, I...",negative
4684,This is the kind of film that might give you a...,positive
1731,This mystery/psychological-thriller is I think...,positive
4742,This isn't the video nasty Night of the Demon ...,positive
4521,"One of my best films ever, maybe because i was...",positive


In [44]:
reviews = reviews.rename(columns={"review": "text", "sentiment": "label"})
reviews['label'] = LabelEncoder().fit_transform(reviews['label'])
reviews.head()

,text,label
6252,"When I first popped in Happy Birthday to Me, I...",0
4684,This is the kind of film that might give you a...,1
1731,This mystery/psychological-thriller is I think...,1
4742,This isn't the video nasty Night of the Demon ...,1
4521,"One of my best films ever, maybe because i was...",1


To help with this task, we're going to use the [Datasets library](https://huggingface.co/docs/datasets/en/index), which allows for working with various types of datasets, from text, to audio, to images.

First, convert the reviews DataFrame into a dataset object by using the [from_pandas function](https://huggingface.co/docs/datasets/v4.8.4/en/package_reference/main_classes#datasets.Dataset.from_pandas).

In [45]:
ds = Dataset.from_pandas(reviews)
ds.features

{'text': Value(dtype='string', id=None),
 'label': Value(dtype='int64', id=None),
 '__index_level_0__': Value(dtype='int64', id=None)}

Once converted, we can perform a train/test split using a method of the Dataset object.

Peform an 80/20 train/test split using the [train_test_split method](https://huggingface.co/docs/datasets/v1.8.0/package_reference/main_classes.html#datasets.Dataset.train_test_split). Save the result back to the same Dataset object.

In [46]:
train_test_split_ds = ds.train_test_split(
    test_size=0.2,
    seed=1,
)
print(train_test_split_ds)

DatasetDict({
    train: Dataset({
        features: ['text', 'label', '__index_level_0__'],
        num_rows: 400
    })
    test: Dataset({
        features: ['text', 'label', '__index_level_0__'],
        num_rows: 100
    })
})


Now, extract the training and test portion into separate Datasets. Name these new datasets train_dataset and test_dataset, respectively.

In [47]:
train_dataset = train_test_split_ds['train']
test_dataset = train_test_split_ds['test']

We're going to be working with a DistilBERT model, which means that we'll need to tokenize our input in the way that DistilBERT expects. For this, we can use the [DistilBertTokenizerFast](https://huggingface.co/docs/transformers/en/model_doc/distilbert?usage=AutoModel#transformers.DistilBertTokenizerFast) tokenizer.

In [48]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

Then we'll apply our tokenizer to the training and test datsets using the map method.

In [49]:
train_dataset = train_dataset.map(lambda df: tokenizer(df['text'], padding="max_length", truncation=True), batched=True)
test_dataset = test_dataset.map(lambda df: tokenizer(df['text'], padding="max_length", truncation=True), batched=True)

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Finally, we'll set the format to torch so that we're working with [PyTorch](https://pytorch.org/) tensors and only extract the columns that we need.

In [50]:
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Now, let's load in the pretrained DistilBERT model.

### Part 1: Fine-tuning All Parameters

In [51]:
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Now, we need to set up a [Trainer](https://huggingface.co/docs/transformers/en/main_classes/trainer) for our model.

First, create a [TraininingArguments](https://huggingface.co/docs/transformers/v4.52.3/en/main_classes/trainer#transformers.TrainingArguments) object. Set num_train_epochs to 5, weight_decay to 0.01, and report_to = 'none'

In [52]:
training_args = TrainingArguments(
    num_train_epochs=5,
    weight_decay=0.01,
    report_to='none'
)

Finally, create a Trainer object using the model, the Training Arguments that you created, and with the train_dataset equal to train_dataset.

In [53]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

Now, use the [train method](https://huggingface.co/docs/transformers/en/main_classes/trainer#transformers.Trainer.train) on your Trainer object.

In [54]:
trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.19719284057617187, metrics={'train_runtime': 110.4908, 'train_samples_per_second': 18.101, 'train_steps_per_second': 2.263, 'total_flos': 264934797312000.0, 'train_loss': 0.19719284057617187, 'epoch': 5.0})

Once the model has been fit, use the [predict method](https://huggingface.co/docs/transformers/en/main_classes/trainer#transformers.Trainer.predict) of the Trainer object to generate a set of predictions on the test_dataset.

In [55]:
predictions = trainer.predict(test_dataset=test_dataset)

Extract the actual test labels and the predicted labels from the predictions. Note that both the true labels and predicted probabilites are contained as an attribute of the predictions.

In [56]:
test_labels = predictions.label_ids
predicted_labels = np.argmax(predictions.predictions, axis=1)

Finally, look the confusion matrix and classification report for these predictions.

In [57]:
confusion_matrix(test_labels, predicted_labels)

array([[47,  3],
       [ 9, 41]])

In [58]:
print(classification_report(test_labels, predicted_labels))

              precision    recall  f1-score   support

           0       0.84      0.94      0.89        50
           1       0.93      0.82      0.87        50

    accuracy                           0.88       100
   macro avg       0.89      0.88      0.88       100
weighted avg       0.89      0.88      0.88       100



### Part 2: Training Only a Subset of the Parameters

Let's first reload the pretrained distilbert model.

Then, we'll make none of the parameters trainable by setting the `requires_grad` attribute to False.

In [59]:
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased")

for param in model.distilbert.parameters():
    param.requires_grad = False

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Then, we'll go back and make the last 2 layers trainable.

In [60]:
for i in [4, 5]:
    for param in model.distilbert.transformer.layer[i].parameters():
        param.requires_grad = True

We'll now set up TrainingArguments and a Trainer as before to train the model.

In [62]:
training_args = TrainingArguments(
    num_train_epochs=5,
    weight_decay=0.01,
    report_to = 'none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.2206337432861328, metrics={'train_runtime': 58.5221, 'train_samples_per_second': 34.175, 'train_steps_per_second': 4.272, 'total_flos': 264934797312000.0, 'train_loss': 0.2206337432861328, 'epoch': 5.0})

How do the results of fine-tuning only the last two layers compare to fine-tuning all layers? How does the training time compare?

## Very similar results

In [63]:
predictions = trainer.predict(test_dataset=test_dataset)

test_labels = predictions.label_ids
predicted_labels = np.argmax(predictions.predictions, axis=1)

print(classification_report(test_labels, predicted_labels))

confusion_matrix(test_labels, predicted_labels)

              precision    recall  f1-score   support

           0       0.82      0.94      0.88        50
           1       0.93      0.80      0.86        50

    accuracy                           0.87       100
   macro avg       0.88      0.87      0.87       100
weighted avg       0.88      0.87      0.87       100



array([[47,  3],
       [10, 40]])

### Part 3: Parameter-Efficient Fine-Tuning

Now, let's see how we can use the [peft library](https://huggingface.co/docs/peft/en/index) to more efficiently fine-tune our model.

First, we'll re-initalize the model.

In [64]:
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Create a [LoraConfig object](https://huggingface.co/docs/peft/package_reference/lora#peft.LoraConfig) were you set the Lora attention dimension to 8, the lora_alpha to 16, the lora_dropout to 0.1, the target_modules to "q_lin" and "v_lin" (these are the "query" and "value" projections), and set the task_type to TaskType.SEQ_CLS.

In [ ]:
#Your Code Here

Now, use the [get_peft_model function](https://huggingface.co/docs/peft/v0.15.0/en/package_reference/peft_model#peft.get_peft_model) to create the Lora model pass in the distilbert model and the LoraConfig object that you created.

In [ ]:
#Your Code Here

How many trainable parameters does the resulting model have? Hint: you can use the [print_trainable_parameters function](https://huggingface.co/docs/peft/v0.15.0/en/package_reference/peft_model#peft.PeftModel.print_trainable_parameters).

In [ ]:
#Your Code Here

We'll again set up the same TrainingArguments and create our Trainer object to grain the model.

In [ ]:
training_args = TrainingArguments(
    num_train_epochs=5,
    weight_decay=0.01,
    report_to = 'none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

How does the performance of the lora model compare to the previous two? How does the training time compare?

In [ ]:
#Your Code Here